In [1]:
import pandas as pd
from sklearn.impute import KNNImputer

# Task 1: Data Integration (Concatenation)

Before cleaning the data, the student records from multiple batches need to be combined into a single dataset for further processing.

## Objective
Merge the two student detail files into one master dataframe.

### Steps

1. **Load the datasets**
   - Import the `pandas` library.
   - Load the following CSV files into separate dataframes:
     - `student_details_batch1.csv`
     - `student_details_batch2.csv`

2. **Concatenate the dataframes**
   - Use `pd.concat()` to vertically combine both dataframes.
   - Store the resulting dataframe in a new variable named `df_master_details`.

3. **Verify the integration**
   - Print the shape of the combined dataframe using:
     ```python
     df_master_details.shape
     ```
   - Confirm that all student records from both batches have been successfully merged into a single dataframe.

In [2]:
batch1 = pd.read_csv('/content/drive/MyDrive/student_details_batch1.csv')

In [3]:
batch1.head()

,StudentID,Age,Monthly_Allowance,Department,Hostel_Status,Study_Hours
0,GLA0001,21.0,7543.397708,Computer Science,NaN,26.8
1,GLA0002,20.0,5540.686576,Mechanical,NaN,35.2
2,GLA0003,NaN,4081.932081,Electrical,Day Scholar,38.2
3,GLA0004,22.0,1263.675179,Civil,Hosteller,10.1
4,GLA0005,20.0,1528.882644,NaN,Day Scholar,37.4


In [4]:
batch2 = pd.read_csv('/content/drive/MyDrive/student_details_batch2.csv')

In [5]:
batch2.head()

,StudentID,Age,Monthly_Allowance,Department,Hostel_Status,Study_Hours
0,GLA0101,NaN,3925.658856,Computer Science,Day Scholar,23.4
1,GLA0102,19.0,NaN,Computer Science,Day Scholar,19.4
2,GLA0103,19.0,NaN,Civil,Day Scholar,39.4
3,GLA0104,19.0,3273.726822,Electrical,Hosteller,8.9
4,GLA0105,20.0,3333.731328,Electrical,NaN,18.9


In [6]:
df_master_details = pd.concat([batch1, batch2], ignore_index=True)
print("Shape of the master dataframe:", df_master_details.shape)

Shape of the master dataframe: (150, 6)


In [7]:
df_master_details.head()

,StudentID,Age,Monthly_Allowance,Department,Hostel_Status,Study_Hours
0,GLA0001,21.0,7543.397708,Computer Science,NaN,26.8
1,GLA0002,20.0,5540.686576,Mechanical,NaN,35.2
2,GLA0003,NaN,4081.932081,Electrical,Day Scholar,38.2
3,GLA0004,22.0,1263.675179,Civil,Hosteller,10.1
4,GLA0005,20.0,1528.882644,NaN,Day Scholar,37.4


# Task 2: Data Cleaning (Handling Missing Values)

The integrated dataset may contain missing values that can affect analysis and model performance. In this task, different imputation techniques will be applied based on the nature of each column.

## Objective
Identify missing values and handle them using appropriate imputation methods.

### Steps

### 2.1 Check Missing Values
- Use `df_master_details.isnull().sum()` to display the number of missing values in each column before performing any imputation.

### 2.2 Mean Imputation (Age)
- Since the **Age** column is approximately normally distributed, calculate its mean.
- Replace all missing values in the **Age** column with the calculated mean.

### 2.3 Median Imputation (Monthly_Allowance)
- The **Monthly_Allowance** column contains extreme outliers.
- Calculate the median of the column.
- Replace all missing values with the median value to avoid the influence of outliers.

### 2.4 Mode Imputation (Department)
- **Department** is a categorical column.
- Calculate the mode (most frequently occurring value).
- Replace all missing values in the column with the mode.

### 2.5 Constant Imputation (Hostel_Status)
- Missing values in **Hostel_Status** indicate that the student has not specified their accommodation.
- Replace all missing values with the constant value **"Not Specified"**.

### 2.6 KNN Imputation (Study_Hours)
- Import `KNNImputer` from `sklearn.impute`.
- Create a subset containing the numerical columns:
  - `Age`
  - `Monthly_Allowance`
  - `Study_Hours`
- Apply KNN Imputation with `n_neighbors=5` to estimate missing values in **Study_Hours** based on similar students.
- Replace the original **Study_Hours** column with the imputed values.

### 2.7 Verify Missing Values
- Use `df_master_details.isnull().sum()` again to confirm that the missing values have been successfully handled.

In [8]:
print("Missing values before imputation:")
print(df_master_details.isnull().sum())

Missing values before imputation:
StudentID             0
Age                  18
Monthly_Allowance    22
Department           12
Hostel_Status        30
Study_Hours          15
dtype: int64


In [9]:
age_mean = df_master_details["Age"].mean()
df_master_details["Age"] = df_master_details["Age"].fillna(age_mean)
allowance_median = df_master_details["Monthly_Allowance"].median()
df_master_details["Monthly_Allowance"] = df_master_details["Monthly_Allowance"].fillna(allowance_median)
department_mode = df_master_details["Department"].mode()[0]
df_master_details["Department"] = df_master_details["Department"].fillna(department_mode)
df_master_details["Hostel_Status"] = df_master_details["Hostel_Status"].fillna("Not Specified")

In [10]:
knn_imputer = KNNImputer(n_neighbors=5)
numerical_cols = df_master_details[["Age", "Monthly_Allowance", "Study_Hours"]]
imputed_values = knn_imputer.fit_transform(numerical_cols)
df_master_details["Study_Hours"] = imputed_values[:, 2]
print("\nMissing values after imputation:")
print(df_master_details.isnull().sum())


Missing values after imputation:
StudentID            0
Age                  0
Monthly_Allowance    0
Department           0
Hostel_Status        0
Study_Hours          0
dtype: int64


# Task 3: Data Integration (Merging)

After cleaning the student details dataset, the next step is to integrate the academic grade records with the student profiles.

## Objective
Merge the student profile dataset with the grades dataset using **StudentID** as the primary key.

### Steps

### 3.1 Load the Grades Dataset
- Load the `student_grades.csv` file into a dataframe named `df_grades`.

### 3.2 Perform an Inner Join
- Merge `df_master_details` and `df_grades` using `StudentID` as the common key.
- Use an **Inner Join** to retain only those students who are present in both datasets.
- Store the merged dataframe in `df_inner_join`.
- Print the shape of the resulting dataframe using:
  ```python
  df_inner_join.shape
  ```
- Observe that only students with both profile details and grade records are included.

### 3.3 Perform a Left Join
- Merge `df_master_details` (left table) with `df_grades` (right table) using `StudentID`.
- Use a **Left Join** so that all student profiles are retained, even if grade records are unavailable.
- Store the merged dataframe in `df_left_join`.
- Print the shape of the resulting dataframe using:
  ```python
  df_left_join.shape
  ```
- Observe that students without grade records contain **NaN** values in the **CGPA** and **Credits_Completed** columns.

In [11]:
df_grades = pd.read_csv("/content/drive/MyDrive/student_grades.csv")

In [12]:
df_inner_join = pd.merge(df_master_details,df_grades,on="StudentID",how="inner")
print("Shape after Inner Join:", df_inner_join.shape)

Shape after Inner Join: (100, 8)


In [13]:
df_inner_join.head()

,StudentID,Age,Monthly_Allowance,Department,Hostel_Status,Study_Hours,CGPA,Credits_Completed
0,GLA0021,19.848485,4146.321443,Computer Science,Not Specified,17.1,8.89,53
1,GLA0022,20.000000,3340.307543,Computer Science,Not Specified,36.4,9.22,119
2,GLA0023,20.000000,4215.439865,Computer Science,Day Scholar,21.6,8.85,113
3,GLA0024,18.000000,3520.132917,Computer Science,Hosteller,28.4,8.26,49
4,GLA0025,19.000000,1295.071184,Computer Science,Hosteller,11.0,6.08,115


In [14]:
df_left_join = pd.merge(df_master_details,df_grades,on="StudentID",how="left")
print("Shape after Left Join:", df_left_join.shape)

Shape after Left Join: (150, 8)


In [15]:
df_left_join.head()

,StudentID,Age,Monthly_Allowance,Department,Hostel_Status,Study_Hours,CGPA,Credits_Completed
0,GLA0001,21.000000,7543.397708,Computer Science,Not Specified,26.8,NaN,NaN
1,GLA0002,20.000000,5540.686576,Mechanical,Not Specified,35.2,NaN,NaN
2,GLA0003,19.848485,4081.932081,Electrical,Day Scholar,38.2,NaN,NaN
3,GLA0004,22.000000,1263.675179,Civil,Hosteller,10.1,NaN,NaN
4,GLA0005,20.000000,1528.882644,Computer Science,Day Scholar,37.4,NaN,NaN


# Task 4: Final Export

After completing data cleaning and merging, save the final master dataset for future analysis.

## Objective
Export the cleaned and left-joined dataset as a CSV file.

### Steps

1. Use the **left-joined** dataframe (`df_left_join`) as the final dataset.
2. Save the dataframe as **`final_student_database.csv`**.
3. Set `index=False` to prevent row indices from being written to the CSV file.
4. Print a confirmation message after the file has been successfully saved.

In [16]:
df_left_join.to_csv("final_student_database.csv", index=False)
print("Final dataset has been successfully saved as 'final_student_database.csv'.")

Final dataset has been successfully saved as 'final_student_database.csv'.
